In [31]:
from youtube_transcript_api import YouTubeTranscriptApi
yt = YouTubeTranscriptApi()
video_id = 'cqRo_xNebkY'
# transcript = yt.fetch(video_id)

# transcript_text=' '.join([t.text for t in transcript])
# print(transcript_text)

In [ ]:
from langchain_core.documents import Document

def get_timestamped_docs(video_id: str) -> list[Document]:
    transcript = yt.fetch(video_id)

    chunks, current_words, current_start, current_end = [], [], None, None

    for entry in transcript:
        words = entry.text.split()
        if current_start is None:
            current_start = entry.start

        current_words.extend(words)
        current_end = entry.start+ entry.duration

        if len(current_words) >= 400:
            chunks.append({
                "text": " ".join(current_words),
                "start": current_start,
                "end": current_end,
            })
            current_words = current_words[-50:]
            current_start = current_end

    if current_words:
        chunks.append({"text": " ".join(current_words), "start": current_start, "end": current_end})

    docs = [
        Document(
            page_content=chunk["text"],
            metadata={
                "video_id": video_id,
                "start_time": chunk["start"],
                "end_time": chunk["end"],
                "yt_url": f"https://youtube.com/watch?v={video_id}&t={int(chunk['start'])}",
            }
        )for chunk in chunks]
    return docs

docs = get_timestamped_docs(video_id)

In [35]:
# # from langchain_text_splitters import RecursiveCharacterTextSplitter
# from langchain_experimental.text_splitter import SemanticChunker
# from langchain_huggingface import HuggingFaceEmbeddings
# embeddings = HuggingFaceEmbeddings(model_name = 'all-MiniLM-L6-v2')
# text_splitter = SemanticChunker(embeddings,breakpoint_threshold_type='percentile')
# # text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200,separators=['\n\n','\n','. ',', ','? ',' '])
# chunks = text_splitter.create_documents([transcript_text])

In [36]:
from langchain_chroma import Chroma
vectorstore = Chroma.from_documents(documents=docs)
retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 3})

In [37]:
retriever.invoke('Furlong')

[Document(id='f334e212-d01b-4f8a-9089-f313032a4ee8', metadata={}, page_content="Let me see. Hey, Chad. Yeah, one sec. What the [\xa0__\xa0] Just give me a sec. You can't move yet, dude. That's like the whole point of ponus, dude. It's an expression. No, it's a reindeer. Ponus may be a bit large for the scale of your machine. Perhaps furlongs are more your speed. Back when the English plowed their own farms instead of colonizing them, they defined a furlong as the distance a team of oxen can plow before getting tired. This works out to be around 200 meters, but that's only true if you raise little [\xa0__\xa0] baby oxen. As I forge mine in a crucible of pain in combat, where only the strongest survive and are easily able to plow 500 m like the badasses they are, the furlong can be broken down into 40 rods, 10 chains, or 7,920 gumballs. It also serves as the basis for an acre, which is one furlong by one chain. And if you really want to work your poor ox into the bone, the amount of land

In [48]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
llm = ChatOllama(model='gemma3:4b',temperature=0)
prompt = PromptTemplate(template='''
                        You are a helpful assistant who can answer user's questions about a YouTube video based on some given context from video transcript.
                        Answer user query only based on the given context. Provide timestamp when the creator mentions this.
                        If the context is not enough, just mention that you don't know.

                        Context:{context}

                        Question:{question}
                        ''',input_variables=['context','question'])

In [49]:
def format_docs(retrieved_docs):
    context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
    return context_text

In [50]:
from langchain_core.runnables import RunnableLambda, RunnableParallel, RunnablePassthrough
parallel_chain = RunnableParallel({
    'context':retriever | RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [51]:
main_chain = parallel_chain | prompt | llm | StrOutputParser()

In [52]:
main_chain.invoke('What is furlong?')

'According to the video transcript, a furlong is the distance a team of oxen can plow before getting tired. It’s approximately 200 meters if you raise “baby oxen” in a crucible of pain. It can also be broken down into 40 rods, 10 chains, or 7,920 gumballs. [00:03:33]'